## 04 - Comparison between Techniques

### Objective
Compare sentiment labels from two models:
- **TextBlob** — lexicon-based, fast, binary-origin with ±0.05 neutral buffer
- **cardiffnlp/twitter-roberta-base-sentiment-latest** — transformer-based, 
  trained on informal text, native three-class output

Where the models disagree, we use **Claude as an LLM judge** to determine 
which model's label is more accurate, giving us a reconciled ground truth 
for notebook 05.

### 1) Setup and Load the Results

In [1]:
!pip install langdetect --quiet

import pandas as pd
import numpy as np
import requests
import time
from pathlib import Path
from langdetect import detect, LangDetectException

ROOT         = Path.cwd().parent
RESULTS_DIR  = ROOT / "data" / "results"

You should consider upgrading via the 'C:\Users\avant\source\airbnb-reviews-nlp\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Load TextBlob and Transformer results for both boroughs.
Both were run on identical samples from `data/samples/` ensuring
the comparison is purely between models, not preprocessing pipelines.

In [3]:
manhattan_textblob    = pd.read_csv(RESULTS_DIR / "manhattan_textblob.csv")
brooklyn_textblob     = pd.read_csv(RESULTS_DIR / "brooklyn_textblob.csv")
manhattan_transformer = pd.read_csv(RESULTS_DIR / "manhattan_transformer.csv")
brooklyn_transformer  = pd.read_csv(RESULTS_DIR / "brooklyn_transformer.csv")

In [4]:
print(f"Manhattan — TextBlob: {len(manhattan_textblob):,} | Transformer: {len(manhattan_transformer):,}")
print(f"Brooklyn  — TextBlob: {len(brooklyn_textblob):,}  | Transformer: {len(brooklyn_transformer):,}")

Manhattan — TextBlob: 14,946 | Transformer: 14,946
Brooklyn  — TextBlob: 14,652  | Transformer: 14,652


### 2) Language Detection Handling

**LIMITATION :** Both TextBlob and `cardiffnlp/twitter-roberta-base-sentiment-latest` are 
English-only models. Non-English reviews will produce unreliable scores 
from both models — TextBlob scores 0.0 polarity (maps to Neutral) and the 
transformer assigns arbitrary sentiment to text it wasn't trained on.

Let's run a language detection on our original samples to compare the affected reviews:

In [6]:
SAMPLES_DIR = ROOT / "data" / "samples"
manhattan_sample_df    = pd.read_csv(SAMPLES_DIR / "manhattan_sample.csv")
brooklyn_sample_df     = pd.read_csv(SAMPLES_DIR / "brooklyn_sample.csv")

In [7]:
from langdetect import detect, LangDetectException

def detect_language(text):
    try:
        return detect(str(text))
    except LangDetectException:
        return 'unknown'

# Run on both samples
manhattan_sample_df['language'] = manhattan_sample_df['reviews'].apply(detect_language)
brooklyn_sample_df['language']  = brooklyn_sample_df['reviews'].apply(detect_language)

print("Manhattan language distribution:")
print(manhattan_sample_df['language'].value_counts().head(10))

print("\nBrooklyn language distribution:")
print(brooklyn_sample_df['language'].value_counts().head(10))

Manhattan language distribution:
language
en       12555
es         707
fr         664
de         256
pt         134
it         129
ko          57
ro          56
nl          51
zh-cn       47
Name: count, dtype: int64

Brooklyn language distribution:
language
en         12992
fr           585
es           305
de           255
nl            74
it            72
ro            41
pt            40
ko            29
unknown       28
Name: count, dtype: int64


We can proceed with ~12K and 13K reviews respectively, so that the comparison accurate. This is documented as a limitation of the analysis. 

A future improvement would be to translate non-English reviews before scoring.

In [ ]:
#TODO: Update the samples to only include English reviews- save as different files.
# Then re-run the analysis and comparison steps.

#### Filtering - 
Since we detected language in our samples dataframe, we can filter both textblob and transformer results - only on matching review_ids for the English language

In [9]:
manhattan_sample_en = manhattan_sample_df[manhattan_sample_df['language'] == 'en'].copy()
brooklyn_sample_en  = brooklyn_sample_df[brooklyn_sample_df['language'] == 'en'].copy()

# Filter transformer results to matching review_ids
manhattan_transformer_en = manhattan_transformer[
    manhattan_transformer['review_id'].isin(manhattan_sample_en['review_id'])
].copy()
brooklyn_transformer_en = brooklyn_transformer[
    brooklyn_transformer['review_id'].isin(brooklyn_sample_en['review_id'])
].copy()


# Filter textblob results to matching review_ids
manhattan_textblob_en = manhattan_textblob[
    manhattan_textblob['review_id'].isin(manhattan_sample_en['review_id'])
].copy()
brooklyn_textblob_en = brooklyn_textblob[
    brooklyn_textblob['review_id'].isin(brooklyn_sample_en['review_id'])
].copy()

print("After English filtering:")
print(f"  Manhattan — TextBlob: {len(manhattan_textblob_en):,} | Transformer: {len(manhattan_transformer_en):,}")
print(f"  Brooklyn  — TextBlob: {len(brooklyn_textblob_en):,}  | Transformer: {len(brooklyn_transformer_en):,}")

After English filtering:
  Manhattan — TextBlob: 12,555 | Transformer: 12,555
  Brooklyn  — TextBlob: 12,992  | Transformer: 12,992


Inadvertently, this filtering changes the % distribution in class labels for the reviews - 

#### Updated distribution for Transformers 

In [50]:
# Check transformer results BEFORE filtering
print("Original transformer distribution (Manhattan):")
print(manhattan_transformer['sentiment_label'].value_counts(normalize=True).round(3))

# Check after English filtering
print("\nAfter English filtering (Manhattan):")
print(manhattan_transformer_en['sentiment_label'].value_counts(normalize=True).round(3))

Original transformer distribution (Manhattan):
sentiment_label
Positive    0.844
Neutral     0.131
Negative    0.025
Name: proportion, dtype: float64

After English filtering (Manhattan):
sentiment_label
Positive    0.956
Negative    0.029
Neutral     0.014
Name: proportion, dtype: float64


In [51]:
# Check transformer results BEFORE filtering
print("Original transformer distribution (Brooklyn):")
print(brooklyn_transformer['sentiment_label'].value_counts(normalize=True).round(3))

# Check after English filtering
print("\nAfter English filtering (Brooklyn):")
print(brooklyn_transformer_en['sentiment_label'].value_counts(normalize=True).round(3))

Original transformer distribution (Brooklyn):
sentiment_label
Positive    0.892
Neutral     0.097
Negative    0.011
Name: proportion, dtype: float64

After English filtering (Brooklyn):
sentiment_label
Positive    0.977
Negative    0.012
Neutral     0.011
Name: proportion, dtype: float64


**Interestingly, the "Neutral" category significantly shrinked from 13.1% to 1.4% for Manhttan, and 9.7% to 1.1% for Brooklyn**

This means the transformer was labeling non-English reviews as Neutral at a much higher rate than English reviews. This is a genuine methodological insight — the model's "uncertainty" when faced with text it wasn't trained on,  manifests as "Neutral" predictions.

#### Updated class distribution for Textblob

In [53]:
# TextBlob distribution before and after English filtering
print("TextBlob distribution BEFORE filtering (Manhattan):")
print(manhattan_textblob['sentiment_label'].value_counts(normalize=True).round(3))

print("\nTextBlob distribution AFTER English filtering (Manhattan):")
print(manhattan_textblob_en['sentiment_label'].value_counts(normalize=True).round(3))


TextBlob distribution BEFORE filtering (Manhattan):
sentiment_label
Positive    0.866
Neutral     0.119
Negative    0.014
Name: proportion, dtype: float64

TextBlob distribution AFTER English filtering (Manhattan):
sentiment_label
Positive    0.963
Neutral     0.025
Negative    0.012
Name: proportion, dtype: float64


In [54]:

print("\nTextBlob distribution BEFORE filtering (Brooklyn):")
print(brooklyn_textblob['sentiment_label'].value_counts(normalize=True).round(3))

print("\nTextBlob distribution AFTER English filtering (Brooklyn):")
print(brooklyn_textblob_en['sentiment_label'].value_counts(normalize=True).round(3))


TextBlob distribution BEFORE filtering (Brooklyn):
sentiment_label
Positive    0.909
Neutral     0.082
Negative    0.009
Name: proportion, dtype: float64

TextBlob distribution AFTER English filtering (Brooklyn):
sentiment_label
Positive    0.977
Neutral     0.017
Negative    0.006
Name: proportion, dtype: float64


**Again, the Neutral class collapsed from  ~12% to 2.5% for Manhtattan and 8.2% to 1.7% for Brooklyn**

**In conclusion: Both techniques were severly inflating the numbers in the "Neutral" category when faced with languages they did not understand.**

### 3) Merge Results

We merge both these filtered results on `review_id`

In [10]:
manhattan_transformer_en.columns

Index(['listing_id', 'review_id', 'date', 'reviews', 'neighbourhood_group',
       'room_type', 'price', 'review_length', 'sentiment_label',
       'score_negative', 'score_neutral', 'score_positive',
       'sentiment_confidence'],
      dtype='object')

In [11]:
manhattan_textblob_en.columns

Index(['listing_id', 'review_id', 'date', 'reviews', 'neighbourhood_group',
       'room_type', 'price', 'review_length', 'polarity', 'subjectivity',
       'sentiment_label', 'subjectivity_label'],
      dtype='object')

In [12]:
def merge_results(textblob_df, transformer_df, borough_name):
    """
    Merges TextBlob and Transformer results on review_id.
    Renames sentiment columns to avoid collision.
    """
    tb = textblob_df[['review_id', 'listing_id', 'reviews',
                       'sentiment_label', 'polarity',
                       'subjectivity', 'subjectivity_label']].rename(
        columns={
            'sentiment_label'     : 'textblob_label',
            'polarity'            : 'textblob_polarity',
            'subjectivity'        : 'textblob_subjectivity',
            'subjectivity_label'  : 'textblob_subjectivity_label'
        }
    )

    tr = transformer_df[['review_id', 'sentiment_label',
                          'score_positive', 'score_neutral',
                          'score_negative', 'sentiment_confidence']].rename(
        columns={
            'sentiment_label'     : 'transformer_label',
            'score_positive'      : 'transformer_score_positive',
            'score_neutral'       : 'transformer_score_neutral',
            'score_negative'      : 'transformer_score_negative',
            'sentiment_confidence': 'transformer_confidence'
        }
    )

    merged = tb.merge(tr, on='review_id', how='inner')
    merged['agreement'] = merged['textblob_label'] == merged['transformer_label']
    merged['borough']   = borough_name

    print(f"\n{borough_name}")
    print(f"  Merged rows : {len(merged):,}")
    print(f"  Dropped     : {len(textblob_df) - len(merged):,}")

    return merged

In [13]:
manhattan_compared = merge_results(manhattan_textblob_en, manhattan_transformer_en, 'Manhattan')
brooklyn_compared  = merge_results(brooklyn_textblob_en,  brooklyn_transformer_en,  'Brooklyn')
combined_compared  = pd.concat([manhattan_compared, brooklyn_compared]).reset_index(drop=True)


Manhattan
  Merged rows : 12,555
  Dropped     : 0

Brooklyn
  Merged rows : 12,992
  Dropped     : 0


In [16]:
COMPARISONS_DIR = ROOT / "data" / "results"/ "comparisons"
COMPARISONS_DIR.mkdir(parents=True, exist_ok=True)

manhattan_compared.to_csv(COMPARISONS_DIR / "manhattan_comparison.csv", index=False)
brooklyn_compared.to_csv(COMPARISONS_DIR / "brooklyn_comparison.csv", index=False)
combined_compared.to_csv(COMPARISONS_DIR / "combined_comparison.csv", index=False)

### 4) Agreement Analysis

#### 4.1 Overall Agreement Rate

In [14]:
for name, df in [('Manhattan', manhattan_compared),
                 ('Brooklyn',  brooklyn_compared),
                 ('Combined',  combined_compared)]:

    agree_rate = df['agreement'].mean()
    print(f"\n{name}")
    print(f"  Agreement rate : {agree_rate:.1%}")
    print(f"  Agreements     : {df['agreement'].sum():,}")
    print(f"  Disagreements  : {(~df['agreement']).sum():,}")


Manhattan
  Agreement rate : 95.3%
  Agreements     : 11,971
  Disagreements  : 584

Brooklyn
  Agreement rate : 97.1%
  Agreements     : 12,621
  Disagreements  : 371

Combined
  Agreement rate : 96.3%
  Agreements     : 24,592
  Disagreements  : 955


#### 4.2 Disagreement Breakdown

Where do the models disagree? The most analytically interesting cases are:
- **TextBlob Positive → Transformer Neutral**: lukewarm reviews TextBlob 
  missed due to lack of negative words — the "quietly dissatisfied" class
- **TextBlob Neutral → Transformer Positive**: reviews TextBlob couldn't 
  score that the transformer correctly identified as positive
- **TextBlob Positive → Transformer Negative**: strongest disagreements — 
  reviews with subtle negativity TextBlob completely missed

In [17]:
disagreements = combined_compared[~combined_compared['agreement']].copy()

print(f"Total disagreements: {len(disagreements):,}")
print(f"\nDisagreement breakdown (TextBlob → Transformer):")
breakdown = disagreements.groupby(
    ['textblob_label', 'transformer_label']
).size().reset_index(name='count').sort_values('count', ascending=False)
print(breakdown.to_string(index=False))

Total disagreements: 955

Disagreement breakdown (TextBlob → Transformer):
textblob_label transformer_label  count
       Neutral          Positive    321
      Positive          Negative    239
      Positive           Neutral    202
       Neutral          Negative    119
      Negative          Positive     48
      Negative           Neutral     26


In [19]:
#TODO: Analyze these categories more

#### 4.3 Confidence on Disagreements

We expect transformer confidence to be lower on disagreement cases —
the model is less certain when the review is ambiguous enough for 
TextBlob to call it differently.

In [18]:
print("Mean transformer confidence:")
print(f"  On agreements    : {combined_compared[combined_compared['agreement']]['transformer_confidence'].mean():.4f}")
print(f"  On disagreements : {combined_compared[~combined_compared['agreement']]['transformer_confidence'].mean():.4f}")

print("\nMean TextBlob polarity:")
print(f"  On agreements    : {combined_compared[combined_compared['agreement']]['textblob_polarity'].mean():.4f}")
print(f"  On disagreements : {combined_compared[~combined_compared['agreement']]['textblob_polarity'].mean():.4f}")

Mean transformer confidence:
  On agreements    : 0.9592
  On disagreements : 0.7226

Mean TextBlob polarity:
  On agreements    : 0.4381
  On disagreements : 0.0866


### 5) LLM as a Judge

We send a sample of disagreement reviews to Claude and ask it to pick 
the correct sentiment label. This gives us a third independent opinion 
to reconcile the two models.

#### 5.1 Claude Judge Function

In [ ]:
#Set your LLM API key - 
os.environ['ANTHROPIC_API_KEY'] = ''  # new key after deleting exposed one


In [34]:
import os
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'  # set if needed

def get_llm_judgment(review_text, textblob_label, transformer_label):
    """
    Sends a disagreement review to Claude and asks it to pick
    the correct sentiment label.
    """

    prompt = f"""You are judging the sentiment of an Airbnb guest review.

Two sentiment models disagreed on this review:
- TextBlob said: {textblob_label}
- Transformer (RoBERTa) said: {transformer_label}

Review:
\"\"\"{review_text}\"\"\"

Pick the most accurate sentiment label from: Positive, Neutral, Negative.
Consider that Airbnb guests tend to write politely even when dissatisfied,
so lukewarm or mildly positive language may indicate Neutral rather than Positive.

Respond in this exact format:
LABEL: <label>
REASON: <one sentence explanation>"""

    try:
        response = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers={
                "Content-Type"      : "application/json",
                "x-api-key"         : os.environ['ANTHROPIC_API_KEY'],
                "anthropic-version" : "2023-06-01"
            },
            json={
                "model"     : "claude-sonnet-4-20250514",
                "max_tokens": 100,
                "messages"  : [{"role": "user", "content": prompt}]
            }
        )

        text   = response.json()['content'][0]['text'].strip()
        lines  = text.split('\n')
        label  = lines[0].replace('LABEL:', '').strip()
        reason = lines[1].replace('REASON:', '').strip() if len(lines) > 1 else ''

        if label not in ['Positive', 'Neutral', 'Negative']:
            label = 'Unknown'

        return {'llm_label': label, 'llm_reasoning': reason}

    except Exception as e:
        return {'llm_label': 'Error', 'llm_reasoning': str(e)}


def run_llm_judge(df, delay=0.5):
    """Runs LLM judgment on a DataFrame of disagreements."""

    results = []
    total   = len(df)

    for i, (_, row) in enumerate(df.iterrows()):
        if i % 20 == 0:
            print(f"  Judging {i}/{total}...")

        judgment = get_llm_judgment(
            row['reviews'],
            row['textblob_label'],
            row['transformer_label']
        )
        results.append(judgment)
        time.sleep(delay)

    judgment_df = pd.DataFrame(results)
    return pd.concat([df.reset_index(drop=True), judgment_df], axis=1)

#### 5.2 Run the LLM As a Judge Function

Test if your API Key is working - 

In [35]:
test = get_llm_judgment(
    "Great location, very clean apartment!",
    "Positive", 
    "Neutral"
)
print(test)

{'llm_label': 'Positive', 'llm_reasoning': 'The review uses clearly positive adjectives ("Great" and "very clean") with exclamation point emphasis, indicating genuine satisfaction rather than polite neutrality.'}


In [27]:
disagreements.shape

(955, 14)

In [36]:

print("Running LLM judge on ALL disagreements...")
judged_disagreements = run_llm_judge(disagreements)

print(f"\nLLM judgment distribution:")
print(judged_disagreements['llm_label'].value_counts())

print(f"\nLLM sided with TextBlob    : {(judged_disagreements['llm_label'] == judged_disagreements['textblob_label']).sum()}")
print(f"LLM sided with Transformer : {(judged_disagreements['llm_label'] == judged_disagreements['transformer_label']).sum()}")
print(f"LLM picked neither         : {((judged_disagreements['llm_label'] != judged_disagreements['textblob_label']) & (judged_disagreements['llm_label'] != judged_disagreements['transformer_label'])).sum()}")

Running LLM judge on ALL disagreements...
  Judging 0/955...
  Judging 20/955...
  Judging 40/955...
  Judging 60/955...
  Judging 80/955...
  Judging 100/955...
  Judging 120/955...
  Judging 140/955...
  Judging 160/955...
  Judging 180/955...
  Judging 200/955...
  Judging 220/955...
  Judging 240/955...
  Judging 260/955...
  Judging 280/955...
  Judging 300/955...
  Judging 320/955...
  Judging 340/955...
  Judging 360/955...
  Judging 380/955...
  Judging 400/955...
  Judging 420/955...
  Judging 440/955...
  Judging 460/955...
  Judging 480/955...
  Judging 500/955...
  Judging 520/955...
  Judging 540/955...
  Judging 560/955...
  Judging 580/955...
  Judging 600/955...
  Judging 620/955...
  Judging 640/955...
  Judging 660/955...
  Judging 680/955...
  Judging 700/955...
  Judging 720/955...
  Judging 740/955...
  Judging 760/955...
  Judging 780/955...
  Judging 800/955...
  Judging 820/955...
  Judging 840/955...
  Judging 860/955...
  Judging 880/955...
  Judging 900/955..

On disagreements, the LLM judge sided with the transformer 74% of the time, validating it as the more accurate model for Airbnb review sentiment analysis. TextBlob's lexicon-based approach was correct only 12% of the time on contested reviews, confirming its limitations with informal, politely-worded text. In 13.5% of cases both models were incorrect, highlighting the inherent ambiguity in Airbnb's grade-inflated review language.

In [37]:
## Save the results 
judged_disagreements.to_csv(COMPARISONS_DIR / "judged_disagreements.csv", index=False)

#### 5.3 Reconcile Disagreements

In [38]:

print("LLM verdict by disagreement type:")
reconciliation = judged_disagreements.groupby(
    ['textblob_label', 'transformer_label', 'llm_label']
).size().reset_index(name='count').sort_values('count', ascending=False)
print(reconciliation.to_string(index=False))

LLM verdict by disagreement type:
textblob_label transformer_label llm_label  count
       Neutral          Positive  Positive    246
      Positive          Negative  Negative    183
      Positive           Neutral   Neutral    120
       Neutral          Negative  Negative    113
       Neutral          Positive   Neutral     52
      Positive           Neutral  Negative     47
      Positive          Negative   Neutral     44
      Negative          Positive  Positive     35
      Positive           Neutral  Positive     34
       Neutral          Positive  Negative     23
      Negative           Neutral   Neutral     13
      Positive          Negative  Positive     12
      Negative           Neutral  Negative     12
      Negative          Positive   Neutral     11
       Neutral          Negative   Neutral      4
       Neutral          Negative  Positive      2
      Negative          Positive  Negative      2
      Positive           Neutral     Error      1
      Negative  

In [39]:
print("\nModel accuracy on disagreements (per type):")
for (tb, tr), group in judged_disagreements.groupby(['textblob_label', 'transformer_label']):
    tb_correct = (group['llm_label'] == group['textblob_label']).sum()
    tr_correct = (group['llm_label'] == group['transformer_label']).sum()
    print(f"\n  TextBlob={tb} vs Transformer={tr} ({len(group)} reviews)")
    print(f"    TextBlob correct    : {tb_correct} ({tb_correct/len(group):.0%})")
    print(f"    Transformer correct : {tr_correct} ({tr_correct/len(group):.0%})")


Model accuracy on disagreements (per type):

  TextBlob=Negative vs Transformer=Neutral (26 reviews)
    TextBlob correct    : 12 (46%)
    Transformer correct : 13 (50%)

  TextBlob=Negative vs Transformer=Positive (48 reviews)
    TextBlob correct    : 2 (4%)
    Transformer correct : 35 (73%)

  TextBlob=Neutral vs Transformer=Negative (119 reviews)
    TextBlob correct    : 4 (3%)
    Transformer correct : 113 (95%)

  TextBlob=Neutral vs Transformer=Positive (321 reviews)
    TextBlob correct    : 52 (16%)
    Transformer correct : 246 (77%)

  TextBlob=Positive vs Transformer=Negative (239 reviews)
    TextBlob correct    : 12 (5%)
    Transformer correct : 183 (77%)

  TextBlob=Positive vs Transformer=Neutral (202 reviews)
    TextBlob correct    : 34 (17%)
    Transformer correct : 120 (59%)


For the most part, LLM adjudged transformers to be the more precise technique for text analysis. However, there are a few interesting categories. 

TextBlob Negative and  Transformer Neutral => Nearly 50-50 split

TextBlob Neutral vs Transformer Negative => Transformers correct 95% of the time. This means Textblob placed them in the low polarity zone unable to correctly place the nuance in the emotion of the review. 

TextBlob Positive vs Transformer Negative => Most critical disagreement => Again, transformers were adjudged correct for a resounding majority (77%)

Textblob Negative vs Transformer Positive => Significant portion likely falls into "neither was correct" category.



#### What is happening where LLM picked neither analysis label?

Are both techniques really picking incorrect labels?

In [40]:
neither = judged_disagreements[
    (judged_disagreements['llm_label'] != judged_disagreements['textblob_label']) & 
    (judged_disagreements['llm_label'] != judged_disagreements['transformer_label'])
].copy()

In [43]:
neither

,review_id,listing_id,reviews,textblob_label,textblob_polarity,textblob_subjectivity,textblob_subjectivity_label,transformer_label,transformer_score_positive,transformer_score_neutral,transformer_score_negative,transformer_confidence,agreement,borough,llm_label,llm_reasoning
4,3768856,12192,Edward's place was perfectly located! That was...,Neutral,-0.007194,0.601667,Mixed,Positive,0.4280,0.3201,0.2520,0.4280,False,Manhattan,Negative,
5,446348891421905132,16821,The Good:<br/>cheap<br/>good location<br/>rela...,Positive,0.138889,0.700000,Subjective,Negative,0.1217,0.4208,0.4575,0.4575,False,Manhattan,Neutral,While the review lists several positive aspect...
18,27976514,106363,"Unluckily, after booking the trip I had to can...",Positive,0.292857,0.657143,Mixed,Negative,0.1938,0.2698,0.5364,0.5364,False,Manhattan,Neutral,While the guest uses positive language about t...
22,26771065,195123,The room was a good size. However it didn't se...,Positive,0.129872,0.598910,Mixed,Neutral,0.0830,0.6229,0.2941,0.6229,False,Manhattan,Negative,Despite starting with one mild positive commen...
26,88593376,457829,"I will start off on a positive note, our host ...",Neutral,0.044449,0.532314,Mixed,Positive,0.8433,0.1113,0.0454,0.8433,False,Manhattan,Negative,Despite starting with positive remarks about t...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
904,433297120983179155,37462923,"Nice house , intrusive neighbors I was warned ...",Positive,0.152381,0.447619,Mixed,Neutral,0.1321,0.5652,0.3027,0.5652,False,Brooklyn,Negative,"Despite starting with ""Nice house,"" the review..."
910,1136269191461994220,41078843,Good place overall. Only issue was that some o...,Positive,0.092381,0.396190,Mixed,Negative,0.2660,0.3444,0.3896,0.3896,False,Brooklyn,Neutral,"While the review starts with ""Good place overa..."
945,831256913439187019,600101578919505892,Parking is free but they only have very few sp...,Positive,0.128000,0.686000,Mixed,Neutral,0.0517,0.5655,0.3828,0.5655,False,Brooklyn,Negative,While the review mentions free parking as a po...
952,1257286390981891668,643473669546971313,A little dingier and a bit smellier than I exp...,Positive,0.170833,0.550000,Mixed,Negative,0.1857,0.4001,0.4142,0.4142,False,Brooklyn,Neutral,"While the review ends with ""great stay with no..."


In [44]:
print(f"Total 'neither' cases: {len(neither)}")
print(f"\nBreakdown:")
print(neither.groupby(
    ['textblob_label', 'transformer_label', 'llm_label']
).size().reset_index(name='count').sort_values('count', ascending=False).to_string(index=False))

print("\nSample reviews where LLM picked neither:\n")
for _, row in neither.head(10).iterrows():
    print(f"Review      : {row['reviews'][:200]}...")
    print(f"TextBlob    : {row['textblob_label']}  |  Transformer: {row['transformer_label']}  |  LLM: {row['llm_label']}")
    print(f"Reason      : {row['llm_reasoning']}")
    print()

Total 'neither' cases: 129

Breakdown:
textblob_label transformer_label llm_label  count
      Positive           Neutral  Negative     47
      Positive          Negative   Neutral     44
       Neutral          Positive  Negative     23
      Negative          Positive   Neutral     11
       Neutral          Negative  Positive      2
      Negative           Neutral  Positive      1
      Positive           Neutral     Error      1

Sample reviews where LLM picked neither:

Review      : Edward's place was perfectly located! That was the good point. Edward was also quite available for us. Some other points are not so good. The room is in a basement but the air is very bad, or you get ...
TextBlob    : Neutral  |  Transformer: Positive  |  LLM: Negative
Reason      : 

Review      : The Good:<br/>cheap<br/>good location<br/>relatively quiet<br/>felt safe<br/>good price<br/>The Bad:<br/>Shower didn't drain properly<br/>Bathroom was just pretty trashed in general<br/>Room was clean...


Clearly, LLM is more context aware, nuanced and sharper in determining the tone for the more ambigous reviews. 

**POTENTIAL LIMITATION**: While running the transformer analysis in `03_Transformer_Analysis.ipynb`, we used "max_length=512" in the tokenizer. This means that it effectively caps every review at 350-400 words (i.e. 512 tokens). However, our LLM comparison takes the entire review text into account. This could attribute to the higher accuracy of the LLM's categorization V/s the transformer's.

### Final Labels for Reviews

In deciding the final labels for reviews, we follow this logic:
1. Agreements (96%) - Pick either Textblob or Transformer label
2. Disagreements - 
    - LLM sided with Transformer (710)
    - LLM sided with Textblob (116)
    - LLM picked neither (129)
    We pick the respective labels.

This logic can be simplified to as follows:
- For agreements, pick the Transformer label, for disagreements pick LLM label

In [ ]:
# Build LLM judgment lookup from disagreements
llm_map = judged_disagreements.set_index('review_id')['llm_label'].to_dict()

# Create final label — LLM where available, transformer otherwise
# LLM Label will be available only for disagreements.
combined_compared['final_label'] = combined_compared.apply(
    lambda row: llm_map.get(row['review_id'], row['transformer_label'])
    if llm_map.get(row['review_id'], 'Error') != 'Error'
    else row['transformer_label'],
    axis=1
)

print("Final label distribution:")
print(combined_compared['final_label'].value_counts(normalize=True).round(3))

print("\nSanity check — how final_label was determined:")
combined_compared['label_source'] = combined_compared.apply(
    lambda row: 'llm_judge' if row['review_id'] in llm_map 
                and llm_map[row['review_id']] != 'Error'
                else 'transformer',
    axis=1
)
print(combined_compared['label_source'].value_counts())

Final label distribution:
final_label
Positive    0.966
Negative    0.021
Neutral     0.013
Name: proportion, dtype: float64

Sanity check — how final_label was determined:
label_source
transformer    24593
llm_judge        954
Name: count, dtype: int64


In [57]:
judged_disagreements['llm_label'].value_counts()

llm_label
Negative    380
Positive    330
Neutral     244
Error         1
Name: count, dtype: int64

In [59]:
# SAVING THE FINAL COMPARISON WITH LLM JUDGMENTS
# Save
combined_compared.to_csv(COMPARISONS_DIR / "final_comparison.csv", index=False)


print("Saved:")
print(f"  final_comparison.csv — {len(combined_compared):,} rows")

Saved:
  final_comparison.csv — 25,547 rows
